In [0]:
import requests
import pandas as pd

# --- 1. EXTRACT ---
API_KEY = "312a547f1bc28626edee7a55ad1860c53cc594726b96ae5f451304610e9ce275" 
# Usamos el sensor 3917 del ejemplo oficial (Ozone) para garantizar datos
SENSOR_ID = "3917" 
url = f"https://api.openaq.org/v3/sensors/{SENSOR_ID}/measurements?limit=1000"
headers = {"X-API-Key": API_KEY}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    data = response.json()

    # --- 2. LOAD (Capa Bronze) ---
    df_pd = pd.DataFrame(data['results'])
    df_bronze = spark.createDataFrame(df_pd)

    # Guardamos en Delta Table
    df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_mediciones_sensor")

    print(f"✅ ¡LOGRADO! Datos del sensor {SENSOR_ID} cargados en Bronze.")
    display(spark.table("bronze_mediciones_sensor"))

except Exception as e:
    print(f"❌ Error: {e}")